# Decision-Making Problem Under Uncertainty

## 1. Optimize under uncertainty

In [1]:
# Imports
from sys import path
path.insert(0, '../src')

from scipy.optimize import linprog
import numpy as np
from problem import c, u_nom, A_ineq, B, bounds

In [3]:
cases = {
    "Deterministic":  0.00,
    "Robust (eps=10%)": 0.10,
    "Robust (eps=20%)": 0.20,
    "Robust (eps=30%)": 0.30,
}

for label, eps in cases.items():
    b = B @ u_nom + eps * np.abs(B) @ u_nom
    result = linprog(c, A_ub=-A_ineq, b_ub=-b, bounds=bounds, method='highs')
    a = result.x
    print(f"{label}")
    print(f"  x* = {a[:3].round(2)},  y* = {a[3:].round(2)},  cost = {result.fun:.2f}\n")

Deterministic
  x* = [10. 47. 10.],  y* = [-0. 22.  0.],  cost = 319.00

Robust (eps=10%)
  x* = [10.  53.7 10. ],  y* = [ 1.  31.2  1. ],  cost = 385.90

Robust (eps=20%)
  x* = [10.  60.4 10. ],  y* = [ 2.  40.4  2. ],  cost = 452.80

Robust (eps=30%)
  x* = [10.  67.1 10. ],  y* = [ 3.  49.6  3. ],  cost = 519.70



## 2. Discussions

The robust solutions are always more expensive than the deterministic one: costs of $385.90$, $452.80$, and $519.70$ vs. $319.00$ for $a^*_{\text{det}}$. This is expected: designing for the worst case over $\mathcal{U}_k$ means installing more capacity than the nominal scenario requires.

The structure of $a^*$ stays consistent across all cases: node 2 holds most of the generation and line $y_2$ carries the surplus to node 1. What changes is the scale. As $\epsilon_k$ grows, $x_2$ and $y_2$ increase to cover the higher worst-case demand, and $y_1$, $y_3$ go from $0$ to small positive values to handle potential shortfalls at nodes 2 and 4.

The cost increase is linear in $\epsilon_k$: each additional $10\%$ raises the total cost by $66.9$ CHF. This follows directly from the structure of the box set: as $\epsilon_k$ grows, the worst-case demand at each node increases proportionally. The three experts are essentially offering different risk tolerances, and the solution shows exactly what each one costs.